# Notebook 2: Federated DAPT (Stage 1)

**Runtime: L4 / T4 / A100.**

## Purpose
Runs the **collaborative** stage: federated domain-adaptive pre-training on each
org's **raw** security text. This stage is where all the organizations collaborate, so this
is the only stage that spends a privacy budget. Output = shared LoRA adapters,
one per experiment, saved to Drive for notebooks 2 & 3 to consume.

## Artifacts
```
FedDAPT/
  corpus/curated_clients/        # INPUT  — raw text per client (from notebook 1)
  corpus/eval/heldout_ids.json   # INPUT  — created by notebook 4 §1; filtered out of training
  adapters/<id>/                 # OUTPUT — LoRA adapter + meta.json per experiment
```
Each adapter folder carries a `meta.json` describing what it is. notebook 4 scans
these and evaluates them; notebook 2 never imports from another notebook.

> ⚠️ Scaffold to run on Colab GPU. The DP mechanism here is a simple
> clip-and-noise DP-FedAvg; **validate the privacy accounting** (roadmap Phase 2)
> before any paper claim, or drop in your Opacus per-client engine.

In [ ]:
!pip -q install -U "transformers>=4.40.0" "accelerate>=0.28.0" "datasets>=2.18.0" \
  "peft>=0.10.0" "bitsandbytes>=0.43.0" sentencepiece "pandas==2.2.2"

In [ ]:
import os, json, math, time, random, copy, glob
from datetime import datetime
from collections import Counter
import numpy as np, torch
from datasets import Dataset, DatasetDict
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          default_data_collator)
from peft import LoraConfig, get_peft_model
from torch.utils.data import DataLoader

import os, tempfile
def _load_dotenv(path='.env'):
    # Minimal .env loader (no dependency); real environment vars take precedence.
    try:
        with open(path) as _f:
            for _line in _f:
                _line = _line.strip()
                if not _line or _line.startswith('#') or '=' not in _line:
                    continue
                _k, _v = _line.split('=', 1)
                os.environ.setdefault(_k.strip(), _v.strip().strip('\'"'))
    except FileNotFoundError:
        pass
def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False
_load_dotenv()   # pull NVD_API_KEY / FEDDAPT_ROOT from .env if present
if _in_colab():
    from google.colab import drive; drive.mount('/content/drive')
    PROJECT_ROOT = os.environ.get('FEDDAPT_ROOT', '/content/drive/MyDrive/FedDAPT')
else:
    # PyCharm / remote GPU: set FEDDAPT_ROOT to your data dir, e.g.  export FEDDAPT_ROOT=/data/fedapt
    PROJECT_ROOT = os.environ.get('FEDDAPT_ROOT', os.path.abspath('./FedDAPT'))
SCRATCH = os.environ.get('FEDDAPT_WORK', os.path.join(tempfile.gettempdir(), 'fedapt_work'))
os.makedirs(PROJECT_ROOT, exist_ok=True); os.makedirs(SCRATCH, exist_ok=True)
print('PROJECT_ROOT =', PROJECT_ROOT, '| SCRATCH =', SCRATCH)
RAW_DIR   = f'{PROJECT_ROOT}/corpus/curated_clients'
EVAL_DIR  = f'{PROJECT_ROOT}/corpus/eval'
ADAPTER_DIR = f'{PROJECT_ROOT}/adapters'
os.makedirs(ADAPTER_DIR, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', device)

In [ ]:
MODEL_ID = 'mistralai/Mistral-7B-v0.1'
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32)
lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=['q_proj','v_proj'],
                         lora_dropout=0.05, bias='none', task_type='CAUSAL_LM')
CONFIG = {
    'learning_rate': 2e-5, 'batch_size': 1, 'max_seq_length': 512,
    'num_rounds': 20, 'local_steps': 10, 'proximal_mu': 0.0,
    'dp_max_grad_norm': 0.5,
    # NOTE: placeholder eps->noise map. Replace with a real accountant (Opacus /
    # dp-accounting) before reporting ε. One number per round, composed over rounds.
    'dp_noise_map': {8: 0.5, 3: 0.8, 1: 1.2},
    'byzantine_trim_ratio': 0.2,   # trims 1 each side at n=6
    'byz_boost': 10.0,
}
print('config loaded')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

def load_fresh_model():
    base = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map='auto')
    return get_peft_model(base, lora_config)

def free_model(m):
    del m; torch.cuda.empty_cache()

def get_lora_state(model):
    # Keyed by parameter name; these are the trainable LoRA leaves.
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if 'lora' in n}

def set_lora_state(model, state):
    with torch.no_grad():
        for n, p in model.named_parameters():
            if n in state: p.copy_(state[n].to(p.dtype).to(p.device))

def state_to_arrays(state):
    keys = sorted(state.keys())
    return [state[k].numpy() for k in keys], keys

def arrays_to_state(arrays, keys):
    return {k: torch.tensor(v) for k, v in zip(keys, arrays)}

def texts_to_dataset_packed(texts, max_length=None):
    # Pack short docs end-to-end (separated by EOS) to avoid wasting the context
    # window on padding — security docs average ~65 words.
    max_length = max_length or CONFIG['max_seq_length']
    ids = []
    for t in texts:
        ids += tokenizer(t, add_special_tokens=False)['input_ids'] + [tokenizer.eos_token_id]
    chunks = [ids[i:i+max_length] for i in range(0, len(ids), max_length)]
    rows = [{'input_ids': c, 'attention_mask': [1]*len(c), 'labels': c[:]} for c in chunks if len(c) >= 16]
    return Dataset.from_list(rows)

def save_adapter(model, exp_id, meta):
    path = f'{ADAPTER_DIR}/{exp_id}'
    model.save_pretrained(path)                      # PEFT adapter weights
    meta = {**meta, 'id': exp_id, 'adapter_path': path, 'created': datetime.now().isoformat()}
    json.dump(meta, open(f'{path}/meta.json', 'w'), indent=2)
    print('saved adapter:', path)
@torch.no_grad()
def lm_perplexity(model, texts, max_docs=200):
    # Held-out raw-text perplexity — the DAPT convergence / validation signal.
    was_training = model.training
    model.eval()
    total_nll, total_tok = 0.0, 0
    for t in texts[:max_docs]:
        ids = tokenizer(t, truncation=True, max_length=CONFIG['max_seq_length'], return_tensors='pt').input_ids.to(device)
        if ids.shape[1] < 2:
            continue
        loss = model(input_ids=ids, labels=ids).loss
        ntok = ids.shape[1] - 1
        total_nll += float(loss) * ntok; total_tok += ntok
    if was_training:
        model.train()
    return float(np.exp(total_nll / max(total_tok, 1)))

print('helpers loaded')

## 1. Load raw-text clients (and exclude the held-out eval docs)
Run **notebook 4 §1 once first** so `corpus/eval/heldout_ids.json` exists — that keeps
the benchmark leakage-free. If it's missing we warn and train on everything.

In [ ]:
client_names = sorted(os.path.splitext(os.path.basename(p))[0]
                      for p in glob.glob(f'{RAW_DIR}/client_*.json'))
print('discovered clients:', client_names)

heldout = set()
hp = f'{EVAL_DIR}/heldout_ids.json'
if os.path.exists(hp):
    heldout = set(json.load(open(hp)))
    print(f'Excluding {len(heldout)} held-out eval docs from training.')
else:
    print('WARNING: no heldout_ids.json — run notebook 4 §1 first to avoid eval leakage.')

def doc_key(t):  # stable identity for a raw doc
    import hashlib; return hashlib.md5(t.encode('utf-8')).hexdigest()[:12]

raw_texts = {}
for n in client_names:
    docs = json.load(open(f'{RAW_DIR}/{n}.json'))
    docs = [d for d in docs if doc_key(d) not in heldout]
    raw_texts[n] = docs
    print(f'{n}: {len(docs)} docs (after held-out filter)')

# Balance clients to the smallest so no client is gradient-starved.
m = min(len(v) for v in raw_texts.values())
raw_texts = {n: random.Random(42).sample(v, m) for n, v in raw_texts.items()}
raw_ds = DatasetDict({n: texts_to_dataset_packed(t) for n, t in raw_texts.items()})
print('balanced to', m, 'docs/client; packed samples:', {n: len(d) for n, d in raw_ds.items()})

lm_val = json.load(open(f'{EVAL_DIR}/lm_val.json')) if os.path.exists(f'{EVAL_DIR}/lm_val.json') else []
print('LM validation docs:', len(lm_val))

## 2. Federated infrastructure (FedAvg / FedProx + Byzantine + DP-FedAvg)

In [ ]:
def fedavg(weights, sizes):
    tot = sum(sizes); w = [s/tot for s in sizes]
    return [sum(w[j]*weights[j][i] for j in range(len(w))) for i in range(len(weights[0]))]

def trimmed_mean(weights, sizes, beta=0.2):
    n = len(weights); k = int(beta*n)
    out = []
    for i in range(len(weights[0])):
        stk = np.sort(np.stack([weights[j][i] for j in range(n)]), axis=0)
        out.append(stk[k:n-k].mean(axis=0) if n-2*k > 0 else stk.mean(axis=0))
    return out

def krum(weights, sizes, f=1):
    n = len(weights)
    flat = [np.concatenate([p.reshape(-1) for p in cw]) for cw in weights]
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            D[i, j] = D[j, i] = np.linalg.norm(flat[i]-flat[j])
    m = max(1, n-f-2)
    scores = [np.sort(D[i])[1:1+m].sum() for i in range(n)]
    return weights[int(np.argmin(scores))]

def dp_privatize(delta_arrays, C, noise_mult):
    # Clip the whole update to L2 norm C, then add Gaussian noise (DP-FedAvg style).
    flat = np.concatenate([d.reshape(-1) for d in delta_arrays])
    norm = np.linalg.norm(flat) + 1e-12
    scale = min(1.0, C/norm)
    return [d*scale + np.random.normal(0, noise_mult*C, size=d.shape) for d in delta_arrays]

def poison_update(new_arrays, g_arrays, attack, boost=10.0, seed=0):
    # Update-level Byzantine attacks applied to a malicious client's LoRA delta.
    delta = [na - ga for na, ga in zip(new_arrays, g_arrays)]
    rng = np.random.default_rng(seed)
    if attack == 'sign_flip':
        delta = [-boost * d for d in delta]                          # reverse + amplify
    elif attack == 'scale':
        delta = [boost * d for d in delta]                           # boosting: dominate the mean
    elif attack == 'gaussian':
        delta = [rng.normal(0, 1.0, size=d.shape).astype(d.dtype) for d in delta]
    return [ga + d for ga, d in zip(g_arrays, delta)]

def poison_dataset(ds):
    # Data-level poison: scramble each packed row's tokens into garbage text.
    from datasets import Dataset
    rnd = random.Random(0); rows = []
    for r in ds:
        ids = list(r['input_ids']); rnd.shuffle(ids)
        rows.append({'input_ids': ids, 'attention_mask': r['attention_mask'], 'labels': ids})
    return Dataset.from_list(rows)

def local_train(model, dataset, global_state, keys, use_fedprox, mu):
    model.train()
    loader = DataLoader(dataset, batch_size=CONFIG['batch_size'], shuffle=True, collate_fn=default_data_collator)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=CONFIG['learning_rate'])
    gref = {n: global_state[n].to(device) for n in global_state} if use_fedprox else None
    losses, step = [], 0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        loss = model(**batch).loss
        if use_fedprox:
            prox = sum(((p - gref[n])**2).sum() for n, p in model.named_parameters() if n in gref)
            loss = loss + (mu/2.0)*prox
        loss.backward(); opt.step(); opt.zero_grad()
        losses.append(float(loss)); step += 1
        if step >= CONFIG['local_steps']: break
    return get_lora_state(model), len(dataset), float(np.mean(losses)) if losses else 0.0

def federated_train(model, client_ds, aggregate_fn, epsilon=float('inf'), mu=0.0, name='exp',
                    val_texts=None, malicious=frozenset(), attack='none'):
    use_dp = math.isfinite(epsilon)
    noise_mult = CONFIG['dp_noise_map'].get(int(epsilon), 1.0) if use_dp else 0.0
    boost = CONFIG.get('byz_boost', 10.0)
    global_state = get_lora_state(model)
    keys = sorted(global_state.keys())
    log = []
    best_ppl, best_state = float('inf'), None
    poisoned = {}                                        # precompute data-poison sets
    if attack == 'data_poison':
        for ci, (cn, ds) in enumerate(client_ds.items()):
            if ci in malicious: poisoned[ci] = poison_dataset(ds)
    tag = f' | malicious={sorted(malicious)} attack={attack}' if malicious else ''
    print(f'--- {name} | agg={aggregate_fn.__name__} eps={epsilon} mu={mu} dp={use_dp}{tag} ---')
    for r in range(CONFIG['num_rounds']):
        weights, sizes, rlosses = [], [], []
        g_arrays, _ = state_to_arrays(global_state)
        for ci, (cname, ds) in enumerate(client_ds.items()):
            is_mal = ci in malicious
            train_ds = poisoned[ci] if (is_mal and attack == 'data_poison') else ds
            set_lora_state(model, global_state)             # each client starts from global
            new_state, n, loss = local_train(model, train_ds, global_state, keys, mu > 0, mu)
            new_arrays, _ = state_to_arrays(new_state)
            if is_mal and attack in ('sign_flip', 'scale', 'gaussian'):
                new_arrays = poison_update(new_arrays, g_arrays, attack, boost, seed=r*100+ci)
            if use_dp:
                delta = [na - ga for na, ga in zip(new_arrays, g_arrays)]
                delta = dp_privatize(delta, CONFIG['dp_max_grad_norm'], noise_mult)
                new_arrays = [ga + d for ga, d in zip(g_arrays, delta)]
            weights.append(new_arrays); sizes.append(n); rlosses.append(loss)
        agg = aggregate_fn(weights, sizes)
        global_state = arrays_to_state(agg, keys)
        rec = {'round': r+1, 'mean_loss': float(np.mean(rlosses))}
        if val_texts:                                       # validation-based round selection
            set_lora_state(model, global_state)
            rec['val_ppl'] = lm_perplexity(model, val_texts)
            if rec['val_ppl'] < best_ppl:
                best_ppl = rec['val_ppl']; best_state = {k: v.clone() for k, v in global_state.items()}
        log.append(rec)
        if (r+1) % 5 == 0 or r == 0:
            msg = f'  round {r+1}/{CONFIG["num_rounds"]} loss={np.mean(rlosses):.4f}'
            if 'val_ppl' in rec: msg += f' val_ppl={rec["val_ppl"]:.2f}'
            print(msg)
    set_lora_state(model, best_state if best_state is not None else global_state)
    if best_state is not None: print(f'  selected best round by val_ppl = {best_ppl:.2f}')
    return log

## 3. Experiment matrix — run, save adapter + meta, auto-skip if done
Start with `fedavg_no_dp`; it's the core federation result. Each run saves a
shared DAPT adapter that notebook 3 can build on and notebook 4 can evaluate.

In [ ]:
EXPERIMENTS = [
    {'exp_id': 'dapt_fedavg_no_dp',      'agg': fedavg,        'epsilon': float('inf'), 'mu': 0.0},
    {'exp_id': 'dapt_fedprox_mu_0p01',   'agg': fedavg,        'epsilon': float('inf'), 'mu': 0.01},
    {'exp_id': 'dapt_fedavg_eps_8',      'agg': fedavg,        'epsilon': 8,            'mu': 0.0},
    {'exp_id': 'dapt_fedavg_eps_3',      'agg': fedavg,        'epsilon': 3,            'mu': 0.0},
    {'exp_id': 'dapt_fedavg_eps_1',      'agg': fedavg,        'epsilon': 1,            'mu': 0.0},
    {'exp_id': 'dapt_krum_no_dp',        'agg': krum,          'epsilon': float('inf'), 'mu': 0.0},
    {'exp_id': 'dapt_trimmed_no_dp',     'agg': trimmed_mean,  'epsilon': float('inf'), 'mu': 0.0},
    {'exp_id': 'dapt_trimmed_eps_8',     'agg': trimmed_mean,  'epsilon': 8,            'mu': 0.0},
]

# ---- Byzantine experiments: 1 of 6 clients malicious x {attacks} x {aggregators} ----
BYZ_MALICIOUS = frozenset({0})
for atk in ['sign_flip', 'scale', 'gaussian', 'data_poison']:
    for aggname, aggfn in [('fedavg', fedavg), ('krum', krum), ('trimmed', trimmed_mean)]:
        EXPERIMENTS.append({'exp_id': f'byz_{aggname}_{atk}', 'agg': aggfn,
                            'epsilon': float('inf'), 'mu': 0.0,
                            'malicious': BYZ_MALICIOUS, 'attack': atk})

TO_RUN = EXPERIMENTS[:1]   # widen the slice as you get GPU time; done ones auto-skip

for exp in TO_RUN:
    path = f'{ADAPTER_DIR}/{exp["exp_id"]}'
    if os.path.exists(f'{path}/meta.json'):
        print('skip (done):', exp['exp_id']); continue
    model = load_fresh_model()
    log = federated_train(model, raw_ds, exp['agg'], exp['epsilon'], exp['mu'], exp['exp_id'],
                          val_texts=lm_val, malicious=exp.get('malicious', frozenset()),
                          attack=exp.get('attack', 'none'))
    save_adapter(model, exp['exp_id'], {
        'stage': 'dapt', 'method': exp['agg'].__name__,
        'epsilon': ('inf' if not math.isfinite(exp['epsilon']) else exp['epsilon']),
        'mu': exp['mu'], 'client': 'global', 'init_from': None,
        'attack': exp.get('attack', 'none'), 'num_malicious': len(exp.get('malicious', [])),
        'round_log': log})
    free_model(model)
print('done this session')

## 2b. Local DAPT (no federation) — produces the **ablation B** adapters
Each org runs DAPT on **its own** raw text with **no aggregation**. notebook 3 then
warm-starts each client's instruction tuning from its *own* local adapter
(`INIT_SOURCES = ['LOCAL']`). The gap C−B is the measured value of federating.

In [ ]:
def train_local_dapt(dataset, total_steps):
    # Straight local training on one client's data, no aggregation.
    model = load_fresh_model()
    loader = DataLoader(dataset, batch_size=CONFIG['batch_size'], shuffle=True, collate_fn=default_data_collator)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=CONFIG['learning_rate'])
    step = 0; done = False
    while not done:
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            loss = model(**batch).loss
            loss.backward(); opt.step(); opt.zero_grad(); step += 1
            if step % 25 == 0: print(f'    step {step}/{total_steps} loss={float(loss):.4f}')
            if step >= total_steps: done = True; break
    return model

# Match the federated compute budget: num_rounds * local_steps total updates.
RUN_LOCAL_DAPT = True
if RUN_LOCAL_DAPT:
    budget = CONFIG['num_rounds'] * CONFIG['local_steps']
    for cname, ds in raw_ds.items():
        exp_id = f'dapt_local_{cname}'
        if os.path.exists(f'{ADAPTER_DIR}/{exp_id}/meta.json'):
            print('skip (done):', exp_id); continue
        print('local DAPT:', cname)
        m = train_local_dapt(ds, budget)
        save_adapter(m, exp_id, {'stage': 'dapt', 'method': 'local', 'epsilon': 'inf',
                                 'mu': 0.0, 'client': cname, 'init_from': None})
        free_model(m)
    print('local DAPT adapters ready (ablation B)')


## Next
- **notebook 3** — optionally instruction-tune on top of any adapter here (opt-in).
- **notebook 4** — evaluate these DAPT adapters directly (the "DAPT-only, no labels" rows).